# Gradient Descent Variants

This notebook accompanies the **ML Viz** lesson on Gradient Descent Variants.

We implement SGD, Momentum, RMSprop, and Adam from scratch, visualize their
trajectories on the 2D Rosenbrock function, inspect Adam's bias-correction step
by step, and plot common learning rate schedules.

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/optimization-ml/01-gradient-descent-variants

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#2e3347',
    'axes.labelcolor':  '#94a3b8',
    'text.color':       'white',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'grid.color':       '#2e3347',
    'grid.alpha':       0.4,
    'legend.facecolor': '#1a1d27',
    'legend.edgecolor': '#2e3347',
})

rng = np.random.default_rng(42)

# Brand colour palette
C_BRAND  = '#818cf8'  # indigo  — SGD
C_TEAL   = '#14b8a6'  # teal    — Momentum
C_YELLOW = '#f59e0b'  # yellow  — RMSprop
C_ROSE   = '#f43f5e'  # rose    — Adam

## Intuition — why plain gradient descent isn't enough

Vanilla gradient descent takes a fixed step downhill, and in a **narrow curved valley**
(the shape of most real loss surfaces) that's a disaster: it bounces across the steep walls
while barely crawling along the floor. Two ideas fix it. **Momentum** accumulates a running
average of past gradients, so consistent directions build up speed and the oscillations
cancel. **Adaptive** methods (RMSprop, Adam) give each parameter its *own* learning rate,
scaled down where gradients are large and up where they're small. **Adam** combines both —
momentum + per-parameter scaling + a bias correction — and is the default optimizer for
deep learning. We implement all four from scratch, race them on the Rosenbrock valley, and
verify against `optax`.

## 1  Optimizer trajectories on the Rosenbrock function

The **Rosenbrock banana** function is the standard stress-test for optimizers:

$$f(x, y) = (1 - x)^2 + 100\,(y - x^2)^2$$

Its global minimum is at $(x, y) = (1, 1)$ inside a narrow curved valley.
Plain SGD zig-zags across the steep valley walls while barely advancing along
the floor; momentum and adaptive methods handle it far better.

We run each optimizer for **200 steps** from the start point $(-1, 1)$
with $\eta = 0.001$.

In [ ]:
# ── Rosenbrock function and gradient ──────────────────────────────────────────
def rosenbrock(x, y):
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(x, y):
    dx = -2 * (1 - x) - 400 * x * (y - x**2)
    dy = 200 * (y - x**2)
    return np.array([dx, dy])

# ── Optimizer implementations ─────────────────────────────────────────────────
def run_sgd(start, steps=200, lr=0.001):
    p = np.array(start, dtype=float)
    path = [p.copy()]
    for _ in range(steps):
        g = rosenbrock_grad(*p)
        p -= lr * g
        path.append(p.copy())
    return np.array(path)

def run_momentum(start, steps=200, lr=0.001, beta=0.9):
    p = np.array(start, dtype=float)
    v = np.zeros(2)
    path = [p.copy()]
    for _ in range(steps):
        g = rosenbrock_grad(*p)
        v = beta * v + (1 - beta) * g
        p -= lr * v
        path.append(p.copy())
    return np.array(path)

def run_rmsprop(start, steps=200, lr=0.001, rho=0.9, eps=1e-8):
    p = np.array(start, dtype=float)
    G = np.zeros(2)
    path = [p.copy()]
    for _ in range(steps):
        g = rosenbrock_grad(*p)
        G = rho * G + (1 - rho) * g**2
        p -= lr * g / (np.sqrt(G) + eps)
        path.append(p.copy())
    return np.array(path)

def run_adam(start, steps=200, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    p = np.array(start, dtype=float)
    m = np.zeros(2)
    v = np.zeros(2)
    path = [p.copy()]
    for t in range(1, steps + 1):
        g = rosenbrock_grad(*p)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g**2
        m_hat = m / (1 - b1**t)
        v_hat = v / (1 - b2**t)
        p -= lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(p.copy())
    return np.array(path)

# ── Run all optimizers ────────────────────────────────────────────────────────
START = [-1.0, 1.0]
paths = {
    'SGD':      run_sgd(START),
    'Momentum': run_momentum(START),
    'RMSprop':  run_rmsprop(START),
    'Adam':     run_adam(START),
}
colors = {
    'SGD': C_BRAND, 'Momentum': C_TEAL, 'RMSprop': C_YELLOW, 'Adam': C_ROSE
}

print("Final losses after 200 steps:")
for name, path in paths.items():
    x, y = path[-1]
    print(f"  {name:10s}  f={rosenbrock(x, y):.4f}  pos=({x:.4f}, {y:.4f})")

# ── Contour background ────────────────────────────────────────────────────────
xs = np.linspace(-2, 2, 400)
ys = np.linspace(-0.5, 3, 400)
XX, YY = np.meshgrid(xs, ys)
ZZ = rosenbrock(XX, YY)

fig, (ax_traj, ax_loss) = plt.subplots(1, 2, figsize=(14, 5.5))

# Trajectory panel
ax_traj.contourf(XX, YY, np.log1p(ZZ), levels=40, cmap='inferno', alpha=0.55)
ax_traj.contour( XX, YY, np.log1p(ZZ), levels=40, colors='white', alpha=0.08, linewidths=0.4)
for name, path in paths.items():
    ax_traj.plot(path[:, 0], path[:, 1], '-', color=colors[name],
                 linewidth=1.6, alpha=0.85, label=name)
    ax_traj.plot(*path[0],  'o', color=colors[name], markersize=5)
    ax_traj.plot(*path[-1], 's', color=colors[name], markersize=5)
ax_traj.scatter([1], [1], s=140, color='white', zorder=10, marker='*', label='Minimum (1,1)')
ax_traj.scatter(*START, s=80, color='#94a3b8', zorder=10, label='Start (-1,1)')
ax_traj.set_xlim(-2, 2); ax_traj.set_ylim(-0.5, 3)
ax_traj.set_xlabel('x'); ax_traj.set_ylabel('y')
ax_traj.set_title('Rosenbrock trajectories (200 steps, lr=0.001)', color='white')
ax_traj.legend(fontsize=9)

# Loss curves panel
for name, path in paths.items():
    losses = [rosenbrock(*pt) for pt in path]
    ax_loss.semilogy(losses, color=colors[name], linewidth=1.8, label=f"{name} final={losses[-1]:.3f}")
ax_loss.set_xlabel('Step'); ax_loss.set_ylabel('f(x, y) — log scale')
ax_loss.set_title('Convergence on Rosenbrock', color='white')
ax_loss.legend(fontsize=9)

plt.tight_layout()
plt.show()

**What to notice:** watch the trajectories and the log-scale loss curves. **SGD** zig-zags
across the valley walls and stalls with the highest final loss; **Momentum** damps the
oscillation and pushes along the floor; **RMSprop** and **Adam** adapt per-dimension and get
closest to the minimum `(1,1)`. On this classic stress-test, the ranking SGD < Momentum <
adaptive is exactly why the field moved to Adam.

## 2  Adam step-by-step: bias correction in action

Adam initialises both moment estimates at zero:
$m_0 = v_0 = 0$. At $t=1$ with $\beta_1 = 0.9$ and true gradient $g$:

$$m_1 = (1-\beta_1)\,g = 0.1\,g \quad\Rightarrow\quad m_1 \text{ is 10\texttimes{} too small!}$$

Bias correction divides by $(1-\beta^t)$ to recover the true scale:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

The cell below runs 5 Adam steps on the scalar loss $L(\theta) = \theta^2$
(gradient $= 2\theta$) and prints $\theta$, $m$, $v$, $\hat{m}$, $\hat{v}$
at every step — watch how $\hat{m}$ and $\hat{v}$ stay close to the true
gradient ($\approx 2$) and squared gradient ($\approx 4$) even at $t=1$.

In [ ]:
def adam_step(theta, grad, m, v, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    """One Adam update. Returns (new_theta, new_m, new_v)."""
    m = b1 * m + (1 - b1) * grad
    v = b2 * v + (1 - b2) * grad**2
    m_hat = m / (1 - b1**t)
    v_hat = v / (1 - b2**t)
    theta = theta - lr * m_hat / (np.sqrt(v_hat) + eps)
    return theta, m, v

print(f"{'t':>3}  {'theta':>10}  {'m (raw)':>12}  {'v (raw)':>12}  "
      f"{'m_hat':>10}  {'v_hat':>10}")
print("-" * 65)

theta, m, v = 1.0, 0.0, 0.0
for t in range(1, 6):
    g = 2.0 * theta  # gradient of theta^2
    theta, m, v = adam_step(theta, g, m, v, t)
    m_hat = m / (1 - 0.9**t)
    v_hat = v / (1 - 0.999**t)
    print(f"{t:>3}  {theta:>10.6f}  {m:>12.6f}  {v:>12.8f}  "
          f"{m_hat:>10.4f}  {v_hat:>10.4f}")

print()
print("Note: m_hat and v_hat stay near the true gradient (~2) and")
print("squared gradient (~4) even at t=1, thanks to bias correction.")

**What to notice:** at `t=1` the raw moment `m` is `0.1×` the true gradient (it started at
0 and only moved 10% of the way), but the **bias-corrected** `m̂ = m/(1−β₁ᵗ)` already sits
near the true gradient `≈2`, and `v̂` near the true squared gradient `≈4`. Without that
`1/(1−βᵗ)` correction Adam would take absurdly tiny first steps — the correction is what lets
it start moving immediately.

## 4. The library way — validate against `optax`

You'd never hand-code Adam in production; you'd use `torch.optim.Adam` or, in JAX,
`optax.adam`. Both implement the exact formula from §2. The cell runs `optax.adam` on the
same Rosenbrock problem (gradients from `jax.grad`) and asserts its trajectory matches our
from-scratch `run_adam`.

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)          # match numpy float64
import jax.numpy as jnp, optax

def ros(pt):
    return (1 - pt[0])**2 + 100 * (pt[1] - pt[0]**2)**2

opt = optax.adam(0.001)                             # same b1=0.9, b2=0.999, eps=1e-8
theta = jnp.array([-1.0, 1.0]); state = opt.init(theta)
optax_path = [np.array(theta)]
for _ in range(200):
    g = jax.grad(ros)(theta)
    updates, state = opt.update(g, state, theta)
    theta = optax.apply_updates(theta, updates)
    optax_path.append(np.array(theta))
optax_path = np.array(optax_path)

print('optax  Adam final:', optax_path[-1].round(5))
print('our    Adam final:', paths['Adam'][-1].round(5))
assert np.allclose(optax_path[-1], paths['Adam'][-1], atol=1e-4), "optax must match our Adam"
print('\nour from-scratch Adam == optax.adam ✓')

**What to notice:** the two Adam trajectories land on the same point to four decimals — our
implementation *is* what `optax.adam` (and `torch.optim.Adam`) does. Understanding the eight
lines of §2 means understanding the optimizer training every modern network.

## 3  Learning rate schedules

A fixed learning rate is rarely optimal. Three common schedules:

| Schedule | Formula |
|----------|---------|
| Constant | $\eta_t = \eta_0$ |
| Step decay | $\eta_t = \eta_0 \cdot \gamma^{\lfloor t/k \rfloor}$ |
| Cosine annealing | $\eta_t = \eta_{\min} + \tfrac{1}{2}(\eta_{\max} - \eta_{\min})\bigl(1 + \cos(\pi t / T)\bigr)$ |

Cosine annealing slows naturally near the minimum, giving the optimizer more
time to fine-tune. Warm restarts (SGDR) periodically reset $t \leftarrow 0$
to escape local minima and explore new regions.

In [ ]:
T = 100  # total epochs
epochs = np.arange(T + 1)

lr_max, lr_min = 0.1, 0.0

def constant_lr(t, T, lr_max, lr_min=0.0):
    return np.full_like(t, lr_max, dtype=float)

def step_decay(t, T, lr_max, lr_min=0.0, gamma=0.5, step=20):
    return lr_max * (gamma ** (t // step))

def cosine_lr_schedule(t, T, lr_max, lr_min=0.0):
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * t / T))

sched_constant = constant_lr(epochs, T, lr_max)
sched_step     = step_decay(epochs, T, lr_max)
sched_cosine   = cosine_lr_schedule(epochs, T, lr_max)

fig, ax = plt.subplots(figsize=(10, 4.5))

ax.plot(epochs, sched_constant, color='#64748b', linewidth=2.0,
        linestyle='--', label='Constant  $\\eta=0.1$')
ax.plot(epochs, sched_step,     color=C_YELLOW, linewidth=2.0,
        label='Step decay  $\\gamma=0.5$, $k=20$')
ax.plot(epochs, sched_cosine,   color=C_TEAL,   linewidth=2.5,
        label='Cosine annealing')

# Annotate cosine formula
mid_t = T // 2
mid_lr = sched_cosine[mid_t]
ax.annotate(
    r'$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max}-\eta_{\min})(1+\cos(\pi t/T))$',
    xy=(mid_t, mid_lr),
    xytext=(mid_t - 28, mid_lr + 0.028),
    color=C_TEAL,
    fontsize=9.5,
    arrowprops=dict(arrowstyle='->', color=C_TEAL, lw=1.2),
)

# Mark endpoints of cosine
ax.scatter([0, T], [sched_cosine[0], sched_cosine[-1]],
           color=C_TEAL, s=60, zorder=10)
ax.text(2, sched_cosine[0] + 0.003, f'$\\eta_{{max}}={lr_max}$',
        color=C_TEAL, fontsize=9)
ax.text(T - 18, sched_cosine[-1] + 0.005, f'$\\eta_{{min}}={lr_min}$',
        color=C_TEAL, fontsize=9)

ax.set_xlabel('Epoch')
ax.set_ylabel('Learning rate $\\eta$')
ax.set_title('Learning rate schedules over 100 epochs', color='white')
ax.set_xlim(0, T)
ax.set_ylim(-0.005, 0.115)
ax.legend(fontsize=10)
ax.grid(True)
plt.tight_layout()
plt.show()

**What to notice:** the constant rate never eases off, step decay drops in discrete cliffs,
and **cosine annealing** glides smoothly to near-zero — giving the optimizer big steps early
(explore) and tiny steps late (fine-tune). Cosine (often with warm restarts) is the default
schedule for training large models today.

## 5. Gotchas & tradeoffs

| | Pros | Cons |
|---|---|---|
| **SGD** | simple, generalizes well, low memory | slow in valleys, LR-sensitive |
| **Momentum** | accelerates, damps oscillation | one more hyperparameter (β) |
| **Adam** | fast, robust default, per-param LR | 2× optimizer memory (m, v), can generalize slightly worse |

- **Learning rate is the master knob.** Too high diverges (even on a convex bowl); too low
  crawls. Adaptive methods are *more* forgiving but not immune.
- **Adam stores two extra values per parameter** (`m` and `v`) — 2× the model size in
  optimizer state, a real cost at billion-parameter scale.
- **Adam can generalize slightly worse than SGD+momentum** on some vision tasks — SGD isn't
  obsolete.
- **Transformers need LR warmup**: Adam's early steps are unstable at high LR, so training
  ramps the rate up before annealing it down.

In [ ]:
# Learning rate too high diverges — even simple SGD blows up on Rosenbrock
def sgd_final(lr, steps=200):
    pt = np.array([-1.0, 1.0])
    for _ in range(steps):
        pt = pt - lr * rosenbrock_grad(*pt)
        if not np.all(np.isfinite(pt)) or np.any(np.abs(pt) > 1e6):
            return 'DIVERGED'
    return f'f={rosenbrock(*pt):.4f}'

for lr in [0.001, 0.002, 0.005]:
    print(f'SGD lr={lr}: {sgd_final(lr)}')
print('\nAdam optimizer state = m and v per parameter -> ~2x model size in memory')

**What to notice:** nudging SGD's learning rate up just a few× turns steady progress into
**divergence** — the valley's steep walls amplify oversized steps. This LR sensitivity is
precisely what adaptive methods (and schedules/warmup) exist to tame, at the cost of extra
optimizer memory.

## Key takeaways

- Plain **SGD** struggles in curved valleys; **momentum** accelerates along consistent
  directions and cancels oscillation.
- **Adaptive** methods (RMSprop/Adam) give each parameter its own effective learning rate;
  **Adam** = momentum + adaptivity + **bias correction** (the `1/(1−βᵗ)` that fixes the
  cold start).
- Our from-scratch Adam matches **`optax.adam`** / `torch.optim.Adam` exactly.
- **Learning-rate schedules** (cosine annealing, warm restarts) trade exploration for
  fine-tuning; transformers add **warmup**.
- Tradeoffs: LR sensitivity, Adam's 2× memory, and SGD's sometimes-better generalization.

**Next:** [Convex Optimization](https://ml-viz-ruby.vercel.app/courses/optimization-ml/02-convex-optimization).

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: each concept is recapped, the code
outline is set, and `# TODO(you)` marks what you fill in. Run the `assert`
cell after each exercise — it passes silently (with a ✅ message) when correct.

### Exercise 1 — Implement one Adam step

Adam maintains two exponential moving averages of the gradient and its square,
then applies bias correction before updating the parameter:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t \qquad\text{(first moment)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2 \qquad\text{(second moment)}$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t} \qquad\text{(bias correction)}$$
$$\theta_t = \theta_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \varepsilon}\,\hat{m}_t \qquad\text{(update)}$$

Implement `adam_update` below. It takes the current parameter `theta`, the
gradient `grad`, the current moment estimates `m` and `v`, and the timestep `t`.

In [ ]:
def adam_update(theta, grad, m, v, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    """One Adam update step.

    Args:
        theta: current parameter value (scalar or array)
        grad:  gradient at theta
        m:     first moment estimate (same shape as theta)
        v:     second moment estimate (same shape as theta)
        t:     current timestep (1-indexed)

    Returns:
        (new_theta, new_m, new_v)
    """
    # TODO(you): update first moment: b1 * m + (1 - b1) * grad
    m = ...

    # TODO(you): update second moment: b2 * v + (1 - b2) * grad**2
    v = ...

    # TODO(you): bias-corrected first moment: m / (1 - b1**t)
    m_hat = ...

    # TODO(you): bias-corrected second moment: v / (1 - b2**t)
    v_hat = ...

    # TODO(you): parameter update: theta - lr * m_hat / (sqrt(v_hat) + eps)
    theta = ...

    return theta, m, v

In [ ]:
# Checks — run me after filling in adam_update above
theta0 = 1.0
m0, v0 = 0.0, 0.0

theta1, m1, v1 = adam_update(theta0, 1.0, m0, v0, 1)
theta2, m2, v2 = adam_update(theta1, 1.0, m1, v1, 2)
theta3, m3, v3 = adam_update(theta2, 1.0, m2, v2, 3)

# theta must strictly decrease each step (gradient is positive, so theta moves left)
assert theta1 < theta0, "theta must decrease after step 1"
assert theta2 < theta1, "theta must decrease after step 2"
assert theta3 < theta2, "theta must decrease after step 3"

# Total movement over 3 steps should be small (lr=0.001)
assert abs(theta3 - theta0) < 0.1, "|theta_3 - theta_0| should be < 0.1 for lr=0.001"

# After one step with grad=1, m and v must both be positive
assert m1 > 0, "first moment m must be positive after one step with positive gradient"
assert v1 > 0, "second moment v must be positive after one step with positive gradient"

# At t=1 with grad=1: m1 = (1-0.9)*1 = 0.1, v1 = (1-0.999)*1 = 0.001  => v1 < m1
assert v1 < m1, (
    f"v ({v1:.6f}) should be less than m ({m1:.6f}) at t=1 "
    "because v tracks squared gradient with b2=0.999 (much slower decay)"
)

# Edge case — learning rate of 0 must never move the parameter
theta_lr0, m_lr0, v_lr0 = adam_update(5.0, 3.0, 0.2, 0.05, 4, lr=0.0)
assert theta_lr0 == 5.0, "lr=0 must leave theta untouched, regardless of m/v/t"

# Edge case — a single optimization step from a fresh (zero) state, checked
# against a hand-derived value: m=0.4, v=0.016 exactly, so
# m_hat=4.0, v_hat=16.0, update = 0.1 * 4.0 / (sqrt(16.0)+eps) = 0.1
theta_1step, m_1step, v_1step = adam_update(2.0, 4.0, 0.0, 0.0, 1, lr=0.1)
assert abs(theta_1step - 1.9) < 1e-6, f"single step should land at theta=1.9, got {theta_1step}"

# Edge case — a single step with zero gradient must not move theta at all
theta_zerograd, _, _ = adam_update(2.0, 0.0, 0.0, 0.0, 1, lr=0.1)
assert theta_zerograd == 2.0, "zero gradient must not move the parameter"

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def adam_update(theta, grad, m, v, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    m = b1 * m + (1 - b1) * grad
    v = b2 * v + (1 - b2) * grad**2
    m_hat = m / (1 - b1**t)
    v_hat = v / (1 - b2**t)
    theta = theta - lr * m_hat / (np.sqrt(v_hat) + eps)
    return theta, m, v
```

</details>

### Exercise 2 — Implement cosine annealing

Cosine annealing smoothly decays the learning rate from $\eta_{\max}$ to
$\eta_{\min}$ over $T$ epochs following a half-cosine curve:

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\!\left(1 + \cos\!\left(\frac{\pi\, t}{T}\right)\right)$$

At $t=0$: $\cos(0) = 1$, so $\eta_0 = \eta_{\max}$.  
At $t=T$: $\cos(\pi) = -1$, so $\eta_T = \eta_{\min}$.  
At $t=T/2$: $\cos(\pi/2) = 0$, so $\eta_{T/2} = \tfrac{1}{2}(\eta_{\max}+\eta_{\min})$.

Implement `cosine_lr` below for a **scalar** epoch `t` (not an array).

In [ ]:
def cosine_lr(t, T, lr_max, lr_min=0.0):
    """Cosine annealing learning rate at epoch t.

    Args:
        t:      current epoch (0-indexed scalar int or float)
        T:      total number of epochs
        lr_max: maximum (starting) learning rate
        lr_min: minimum (ending) learning rate (default 0.0)

    Returns:
        learning rate at epoch t (scalar float)
    """
    # TODO(you): apply the cosine annealing formula
    # hint: use np.cos and np.pi
    return ...

In [ ]:
# Checks — run me after filling in cosine_lr above
assert abs(cosine_lr(0,   100, 0.1) - 0.1)  < 1e-9, "at t=0 lr should equal lr_max"
assert abs(cosine_lr(100, 100, 0.1) - 0.0)  < 1e-9, "at t=T lr should equal lr_min (0.0)"
assert abs(cosine_lr(50,  100, 0.1) - 0.05) < 1e-9, "at t=T/2 lr should be halfway between max and min"

# Non-zero lr_min
assert abs(cosine_lr(0,   100, 0.1, lr_min=0.01) - 0.1)  < 1e-9
assert abs(cosine_lr(100, 100, 0.1, lr_min=0.01) - 0.01) < 1e-9
assert abs(cosine_lr(50,  100, 0.1, lr_min=0.01) - 0.055) < 1e-9

# Edge case — a single-step schedule (T=1): epoch 0 is lr_max, epoch 1 (=T) is lr_min
assert abs(cosine_lr(0, 1, 0.2) - 0.2) < 1e-9, "T=1, epoch 0 should equal lr_max"
assert abs(cosine_lr(1, 1, 0.2) - 0.0) < 1e-9, "T=1, epoch 1 (=T) should equal lr_min"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def cosine_lr(t, T, lr_max, lr_min=0.0):
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * t / T))
```

</details>

---
## 🧪 Extra practice — optimizer implementation bank

The exercises above cover Adam and cosine annealing in depth. The
[Gradient Descent Optimizers wiki page](https://ml-viz-ruby.vercel.app/wiki/gradient-descent-optimizers)
walks through the full optimizer family; here we implement six more of them exactly
as posed by [DML-OpenProblem](https://github.com/Open-Deep-ML/DML-OpenProblem):

- DML **145** — Adagrad optimizer
- DML **146** — Momentum optimizer
- DML **148** — Adamax optimizer
- DML **149** — Adadelta optimizer
- DML **150** — Nesterov Accelerated Gradient (NAG) optimizer
- DML **87** — Adam optimizer, the **per-step stateful** signature
  `(parameter, grad, m, v, t) -> (parameter, m, v)` — the caller stores `m`, `v`,
  `t` between calls and invokes the function once per optimization step. (This is
  a different framing from DML 49, the loop-based Adam already implemented in the
  `neural-networks` course notebook — same math, different calling convention.)

All six share that per-step convention: pass in the current parameter, gradient,
and whatever running state the optimizer needs (`G`, `velocity`, `(m, u)`,
`(u, v)`, ...), get back the updated parameter and updated state. That symmetry
is what lets one toy quadratic loss $L(\theta) = \theta^2$
(so $\nabla L(\theta) = 2\theta$, minimum at $\theta = 0$) drive all six
through the same `run_optimizer` harness below.

Fill in the `# TODO(you)` lines (the update rule for each optimizer), then run
the assert cell.

In [ ]:
# ── Optimizer bank: Adagrad (145), Momentum (146), Adamax (148),
#    Adadelta (149), Nesterov/NAG (150), Adam per-step (87) ──────────────────

def adagrad_optimizer(parameter, grad, G, learning_rate=0.01, epsilon=1e-8):
    """DML 145. G accumulates squared gradients; the effective step size
    shrinks per-parameter as G grows."""
    # TODO(you): G = G + grad**2
    G = ...
    # TODO(you): parameter = parameter - learning_rate * grad / (sqrt(G) + epsilon)
    parameter = ...
    return parameter, G


def momentum_optimizer(parameter, grad, velocity, learning_rate=0.01, momentum=0.9):
    """DML 146. velocity is a decayed running sum of learning-rate-scaled
    gradients — this is the "classical momentum" update."""
    # TODO(you): velocity = momentum * velocity + learning_rate * grad
    velocity = ...
    # TODO(you): parameter = parameter - velocity
    parameter = ...
    return parameter, velocity


def adamax_optimizer(parameter, grad, m, u, t, learning_rate=0.002,
                      beta1=0.9, beta2=0.999, epsilon=1e-8):
    """DML 148. Like Adam, but the second-moment tracker u is a running
    infinity norm (elementwise max) instead of an exponential average of
    squared gradients."""
    m = beta1 * m + (1 - beta1) * grad
    # TODO(you): u = max(beta2 * u, abs(grad))  (np.maximum handles array inputs)
    u = ...
    m_hat = m / (1 - beta1 ** t)
    # TODO(you): parameter = parameter - (learning_rate / (u + epsilon)) * m_hat
    parameter = ...
    return parameter, m, u


def adadelta_optimizer(parameter, grad, u, v, rho=0.95, epsilon=1e-6):
    """DML 149. No explicit learning rate: u tracks squared gradients, v
    tracks squared *parameter updates*, and their ratio calibrates the step
    size — Adadelta adapts even the learning rate itself away."""
    u = rho * u + (1 - rho) * grad ** 2
    # TODO(you): delta = sqrt(v + epsilon) / sqrt(u + epsilon) * grad
    delta = ...
    parameter = parameter - delta
    # TODO(you): v = rho * v + (1 - rho) * delta**2
    v = ...
    return parameter, u, v


def nag_optimizer(parameter, grad_fn, velocity, learning_rate=0.01, momentum=0.9):
    """DML 150. "Look-ahead" momentum: the gradient is evaluated at
    parameter - momentum * velocity, not at parameter itself."""
    # TODO(you): lookahead = parameter - momentum * velocity
    lookahead = ...
    grad = grad_fn(lookahead)
    velocity = momentum * velocity + learning_rate * grad
    parameter = parameter - velocity
    return parameter, velocity


def adam_optimizer(parameter, grad, m, v, t, learning_rate=0.001,
                    beta1=0.9, beta2=0.999, epsilon=1e-8):
    """DML 87 — Adam's per-step stateful form: one call per optimization
    step, with m, v, t threaded through by the caller (contrast with the
    loop-based Adam in the neural-networks course notebook)."""
    m = beta1 * m + (1 - beta1) * grad
    v = beta2 * v + (1 - beta2) * grad ** 2
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    # TODO(you): parameter = parameter - learning_rate * m_hat / (sqrt(v_hat) + epsilon)
    parameter = ...
    return parameter, m, v


# ── Shared toy quadratic-loss harness: L(theta) = theta**2, grad(theta) = 2*theta ──
def quad_grad(theta):
    return 2 * theta


def run_optimizer(step_fn, theta0, n_steps=150):
    """Drive a scalar parameter toward the minimum of L(theta) = theta**2 for
    n_steps calls of step_fn(theta) -> new_theta. Returns the full trajectory."""
    theta = theta0
    trajectory = [theta]
    for _ in range(n_steps):
        theta = step_fn(theta)
        trajectory.append(theta)
    return trajectory


def make_adagrad_step(learning_rate=0.5):
    G = 0.0
    def step(theta):
        nonlocal G
        new_theta, G = adagrad_optimizer(theta, quad_grad(theta), G, learning_rate)
        return new_theta
    return step


def make_momentum_step(learning_rate=0.1, momentum=0.9):
    velocity = 0.0
    def step(theta):
        nonlocal velocity
        new_theta, velocity = momentum_optimizer(
            theta, quad_grad(theta), velocity, learning_rate, momentum)
        return new_theta
    return step


def make_adamax_step(learning_rate=0.2):
    m, u, t = 0.0, 0.0, 0
    def step(theta):
        nonlocal m, u, t
        t += 1
        new_theta, m, u = adamax_optimizer(theta, quad_grad(theta), m, u, t, learning_rate)
        return new_theta
    return step


def make_adadelta_step():
    u, v = 0.0, 0.0
    def step(theta):
        nonlocal u, v
        new_theta, u, v = adadelta_optimizer(theta, quad_grad(theta), u, v)
        return new_theta
    return step


def make_nag_step(learning_rate=0.1, momentum=0.9):
    velocity = 0.0
    def step(theta):
        nonlocal velocity
        new_theta, velocity = nag_optimizer(theta, quad_grad, velocity, learning_rate, momentum)
        return new_theta
    return step


def make_adam_step(learning_rate=0.3):
    m, v, t = 0.0, 0.0, 0
    def step(theta):
        nonlocal m, v, t
        t += 1
        new_theta, m, v = adam_optimizer(theta, quad_grad(theta), m, v, t, learning_rate)
        return new_theta
    return step

In [ ]:
# Checks — run me after filling in the six optimizers above

# 1) Single-step correctness against DML-OpenProblem reference values
#    (tolerance-based: DML rounds its own outputs to 5 decimals)
p, G = adagrad_optimizer(1., 0.5, 1., 0.01, 1e-8)
assert abs(p - 0.99553) < 1e-4 and abs(G - 1.25) < 1e-4, "adagrad single-step mismatch"

p, vel = momentum_optimizer(1., 0.1, 0.5, 0.01, 0.9)
assert abs(p - 0.549) < 1e-4 and abs(vel - 0.451) < 1e-4, "momentum single-step mismatch"

p, m, u = adamax_optimizer(1., 0.1, 1., 1., 1, 0.002, 0.9, 0.999, 1e-8)
assert abs(p - 0.98178) < 1e-4 and abs(m - 0.91) < 1e-4 and abs(u - 0.999) < 1e-4, \
    "adamax single-step mismatch"

p, u2, v2 = adadelta_optimizer(1., 0.5, 1., 1., 0.95, 1e-6)
assert abs(p - 0.49035) < 1e-3 and abs(u2 - 0.9625) < 1e-4 and abs(v2 - 0.96299) < 1e-4, \
    "adadelta single-step mismatch"

def _identity_grad(x):   # matches DML's tests.json gradient_function for a scalar x
    return x
p, vel2 = nag_optimizer(1., _identity_grad, 0.5, 0.01, 0.9)
assert abs(p - 0.5445) < 1e-4 and abs(vel2 - 0.4555) < 1e-4, "nag single-step mismatch"

p, m2, v3 = adam_optimizer(1.0, 0.1, 0.0, 0.0, 1)
assert abs(p - 0.999) < 1e-3 and abs(m2 - 0.01) < 1e-4 and abs(v3 - 1e-5) < 1e-6, \
    "adam single-step mismatch"

# 2) Edge case — learning rate of 0 must never move the parameter
p0, _ = momentum_optimizer(3.0, 10.0, 0.0, learning_rate=0.0, momentum=0.9)
assert p0 == 3.0, "lr=0 must not move the parameter (momentum)"
p0b, _ = adagrad_optimizer(3.0, 10.0, 0.0, learning_rate=0.0)
assert p0b == 3.0, "lr=0 must not move the parameter (adagrad)"

# 3) Edge case — a single optimization step from a fresh (zero) state
p1, m1, v1 = adam_optimizer(2.0, 4.0, 0.0, 0.0, 1, learning_rate=0.1)
assert abs(p1 - 1.9) < 1e-6, f"one Adam step with these inputs should land at theta=1.9, got {p1}"

# 4) End-to-end — every optimizer, run through the shared harness, converges
#    toward the minimum theta=0 (adadelta is parameter-free and adapts slowly,
#    so it gets a longer budget)
configs = [
    ("adagrad", make_adagrad_step(), 150),
    ("momentum", make_momentum_step(), 150),
    ("adamax", make_adamax_step(), 150),
    ("adadelta", make_adadelta_step(), 2000),
    ("nag", make_nag_step(), 150),
    ("adam", make_adam_step(), 150),
]
for name, step_fn, n_steps in configs:
    trajectory = run_optimizer(step_fn, theta0=5.0, n_steps=n_steps)
    assert abs(trajectory[-1]) < abs(trajectory[0]), f"{name} should move theta toward 0"
    assert abs(trajectory[-1]) < 1.0, f"{name} should get close to the minimum (got {trajectory[-1]:.4f})"

print("✅ Optimizer bank passed")

<details>
<summary>💡 Show solution</summary>

```python
def adagrad_optimizer(parameter, grad, G, learning_rate=0.01, epsilon=1e-8):
    G = G + grad**2
    parameter = parameter - learning_rate * grad / (np.sqrt(G) + epsilon)
    return parameter, G

def momentum_optimizer(parameter, grad, velocity, learning_rate=0.01, momentum=0.9):
    velocity = momentum * velocity + learning_rate * grad
    parameter = parameter - velocity
    return parameter, velocity

def adamax_optimizer(parameter, grad, m, u, t, learning_rate=0.002,
                      beta1=0.9, beta2=0.999, epsilon=1e-8):
    m = beta1 * m + (1 - beta1) * grad
    u = np.maximum(beta2 * u, np.abs(grad))
    m_hat = m / (1 - beta1 ** t)
    parameter = parameter - (learning_rate / (u + epsilon)) * m_hat
    return parameter, m, u

def adadelta_optimizer(parameter, grad, u, v, rho=0.95, epsilon=1e-6):
    u = rho * u + (1 - rho) * grad ** 2
    delta = np.sqrt(v + epsilon) / np.sqrt(u + epsilon) * grad
    parameter = parameter - delta
    v = rho * v + (1 - rho) * delta ** 2
    return parameter, u, v

def nag_optimizer(parameter, grad_fn, velocity, learning_rate=0.01, momentum=0.9):
    lookahead = parameter - momentum * velocity
    grad = grad_fn(lookahead)
    velocity = momentum * velocity + learning_rate * grad
    parameter = parameter - velocity
    return parameter, velocity

def adam_optimizer(parameter, grad, m, v, t, learning_rate=0.001,
                    beta1=0.9, beta2=0.999, epsilon=1e-8):
    m = beta1 * m + (1 - beta1) * grad
    v = beta2 * v + (1 - beta2) * grad ** 2
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    parameter = parameter - learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)
    return parameter, m, v
```

Adagrad and Adadelta both track squared gradients but diverge in how they use
that history: Adagrad divides by its square root directly (so the effective
rate only ever shrinks — it can stall on long runs), while Adadelta rescales
by the ratio of past-update magnitude to past-gradient magnitude, which is why
it needs no learning rate at all. Adamax swaps Adam's second-moment *average*
for a running *max* (the $\ell_\infty$ norm), which makes it robust to a single
huge gradient spike. NAG's only change from classical momentum is *where* the
gradient is evaluated — at the look-ahead point instead of the current one —
which is enough to measurably speed up convergence on curved loss surfaces.

</details>